In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import itertools
import copy
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. DATASET DEFINITION
# ==========================================
class SyntheticMaaSDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(csv_file)
        
        df['Age_norm'] = (df['Age'] - df['Age'].mean()) / df['Age'].std()
        df['Income_norm'] = (df['Income'] - df['Income'].mean()) / df['Income'].std()
        
        self.z = torch.tensor(df[['Age_norm', 'Income_norm']].values, dtype=torch.float32)
        self.x = torch.tensor(df[['Time_Car', 'Cost_Car', 'Time_PT', 'Cost_PT', 'Time_MaaS', 'Cost_MaaS']].values, dtype=torch.float32)
        self.choice = torch.tensor(df['Choice'].values, dtype=torch.long)

    def __len__(self):
        return len(self.choice)

    def __getitem__(self, idx):
        return self.x[idx], self.z[idx], self.choice[idx]

# ==========================================
# 2. NETWORK ARCHITECTURE
# ==========================================
class F_DNN(nn.Module):
    def __init__(self, x_dim=6, z_dim=2, hidden_dim=20, num_choices=3):
        super(F_DNN, self).__init__()
        
        self.mlp = nn.Sequential(
            nn.Linear(x_dim + z_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, num_choices)
        )

    def forward(self, x, z):
        combined_input = torch.cat((x, z), dim=1) 
        choice_logits = self.mlp(combined_input)
        return choice_logits

# ==========================================
# 3. TRAINING ENGINE
# ==========================================
def run_single_fdnn_training(train_loader, test_loader, hidden_dim, lr, epochs=100, seed=42, verbose=False):
    """Executes a single training session for F-DNN and returns optimal validation metrics."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = F_DNN(hidden_dim=hidden_dim).to(device)   
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction='sum')
    
    best_nll = float('inf')
    best_acc = 0.0
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        for x, z, choice in train_loader:
            x, z, choice = x.to(device), z.to(device), choice.to(device)
            
            optimizer.zero_grad()
            choice_logits = model(x, z)
           
            loss = nn.CrossEntropyLoss()(choice_logits, choice) 
            loss.backward()
            optimizer.step()

        model.eval()
        correct, total_samples, total_nll = 0, 0, 0.0
        with torch.no_grad():
             for tx, tz, tc in test_loader:
                 tx, tz, tc = tx.to(device), tz.to(device), tc.to(device)
                 t_logits = model(tx, tz)
                 t_preds = torch.argmax(t_logits, dim=1)
                 
                 correct += (t_preds == tc).sum().item()
                 total_nll += criterion(t_logits, tc).item()
                 total_samples += tc.size(0)
                 
        current_acc = correct / total_samples
        current_nll = total_nll / total_samples

        if current_nll < best_nll:
            best_nll = current_nll
            best_acc = current_acc
            best_model_state = copy.deepcopy(model.state_dict())
            
    if verbose:
        print(f"  [Seed {seed}] Best NLL: {best_nll:.4f}, Best Acc: {best_acc*100:.2f}%")
        
    return best_nll, best_acc, best_model_state

# ==========================================
# 4. ROBUSTNESS EVALUATION
# ==========================================
def run_fdnn_robustness(csv_file='synthetic_maas_dataset.csv', batch_size=200):
    print(f"Starting F-DNN Robustness Evaluation on {device}...\n")
    
    dataset = SyntheticMaaSDataset(csv_file)
    dataset_size = len(dataset)
    indices = list(range(dataset_size))
    np.random.seed(42) 
    np.random.shuffle(indices) 
    split = int(np.floor(0.2 * dataset_size))
    train_indices, test_indices = indices[split:], indices[:split]

    train_sampler = torch.utils.data.SubsetRandomSampler(train_indices)
    train_loader = DataLoader(dataset, batch_size=batch_size, sampler=train_sampler)
    test_dataset = torch.utils.data.Subset(dataset, test_indices)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(" 50 Independent Runs for Robustness (F-DNN) ===")
    num_runs = 50
    results_nll = []
    results_acc = []
    
    absolute_best_nll = float('inf')
    absolute_best_state = None
    
    for i in range(num_runs):
        seed = i 
        nll, acc, model_state = run_single_fdnn_training(train_loader, test_loader, 
                                                         hidden_dim=20, 
                                                         lr=0.001, 
                                                         epochs=100, seed=seed, verbose=True)
        results_nll.append(nll)
        results_acc.append(acc)
        
        if nll < absolute_best_nll:
            absolute_best_nll = nll
            absolute_best_state = copy.deepcopy(model_state)

    mean_nll, std_nll = np.mean(results_nll), np.std(results_nll)
    mean_acc, std_acc = np.mean(results_acc) * 100, np.std(results_acc) * 100
    
    print("\n==============================================")
    print("=== FINAL ROBUSTNESS RESULTS (F-DNN) ===")
    print("==============================================")
    print(f"Model: Pure Black-Box F-DNN")
    print(f"Test Accuracy: {mean_acc:.2f}% ± {std_acc:.2f}%")
    print(f"Test NLL:      {mean_nll:.4f} ± {std_nll:.4f}")
    print("==============================================")

    torch.save(absolute_best_state, 'f_dnn_synthetic_best_run_overall.pth')
    print("The weights of the single best run have been saved to 'f_dnn_synthetic_best_run_overall.pth'.")

if __name__ == "__main__":
    run_fdnn_robustness()

Starting F-DNN Robustness Evaluation on cuda...

 50 Independent Runs for Robustness (F-DNN) ===
  [Seed 0] Best NLL: 0.7754, Best Acc: 64.65%
  [Seed 1] Best NLL: 0.7937, Best Acc: 63.60%
  [Seed 2] Best NLL: 0.7763, Best Acc: 64.05%
  [Seed 3] Best NLL: 0.8378, Best Acc: 61.45%
  [Seed 4] Best NLL: 0.7691, Best Acc: 65.10%
  [Seed 5] Best NLL: 0.7741, Best Acc: 64.20%
  [Seed 6] Best NLL: 0.7744, Best Acc: 64.90%
  [Seed 7] Best NLL: 0.8454, Best Acc: 61.50%
  [Seed 8] Best NLL: 0.7758, Best Acc: 64.65%
  [Seed 9] Best NLL: 0.8431, Best Acc: 61.65%
  [Seed 10] Best NLL: 0.7753, Best Acc: 64.70%
  [Seed 11] Best NLL: 0.8416, Best Acc: 62.25%
  [Seed 12] Best NLL: 0.8037, Best Acc: 63.20%
  [Seed 13] Best NLL: 0.8475, Best Acc: 61.10%
  [Seed 14] Best NLL: 0.8340, Best Acc: 61.95%
  [Seed 15] Best NLL: 0.8460, Best Acc: 60.50%
  [Seed 16] Best NLL: 0.7788, Best Acc: 63.35%
  [Seed 17] Best NLL: 0.7837, Best Acc: 64.25%
  [Seed 18] Best NLL: 0.8418, Best Acc: 61.55%
  [Seed 19] Best NLL